# Data Exploration: Fake News Detection

This notebook explores the fake news dataset and prepares it for model training.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import sys
sys.path.insert(0, '../backend')

# Load data
from src.data_loader import FakeNewsDataLoader
from src.preprocessor import TextPreprocessor

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Dataset

In [ ]:
# Load the dataset
loader = FakeNewsDataLoader()
data = loader.load_data()

print(f"Total samples: {len(data)}")
print(f"\nColumns: {data.columns.tolist()}")
print(f"\nFirst few rows:")
data.head()

## 2. Class Distribution

In [ ]:
# Class distribution
label_counts = data['label'].value_counts()
print("Class Distribution:")
print(label_counts)
print(f"\nPercentage:")
print(data['label'].value_counts(normalize=True) * 100)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
label_counts.plot(kind='bar', ax=axes[0], color=['#ef4444', '#22c55e'])
axes[0].set_title('News Classification Distribution')
axes[0].set_ylabel('Count')
axes[0].set_xlabel('Label')

label_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['#ef4444', '#22c55e'])
axes[1].set_title('Class Distribution %')
plt.tight_layout()
plt.show()

## 3. Text Statistics

In [ ]:
# Text length statistics
data['text_length'] = data['text'].str.len()
data['word_count'] = data['text'].str.split().str.len()

print("Text Length Statistics:")
print(data['text_length'].describe())
print("\nWord Count Statistics:")
print(data['word_count'].describe())

# Compare by label
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

data.boxplot(column='text_length', by='label', ax=axes[0])
axes[0].set_title('Text Length by Label')
axes[0].set_ylabel('Character Count')

data.boxplot(column='word_count', by='label', ax=axes[1])
axes[1].set_title('Word Count by Label')
axes[1].set_ylabel('Word Count')

plt.tight_layout()
plt.show()

## 4. Preprocessing & Tokenization

In [ ]:
preprocessor = TextPreprocessor()

# Sample preprocessing
sample_text = data['text'].iloc[0]
print("Original text:")
print(sample_text[:200])
print("\nPreprocessed text:")
processed = preprocessor.preprocess(sample_text)
print(processed[:200])

# Apply preprocessing to all
data['processed_text'] = data['text'].apply(preprocessor.preprocess)
print("\nPreprocessing completed!")

## 5. Vocabulary Analysis

In [ ]:
# Build vocabulary
from collections import Counter

all_words = []
for text in data['processed_text']:
    all_words.extend(text.split())

word_freq = Counter(all_words)
print(f"Total unique words: {len(word_freq)}")
print(f"\nTop 20 most common words:")
for word, count in word_freq.most_common(20):
    print(f"  {word}: {count}")

# Visualize
words, freqs = zip(*word_freq.most_common(15))
plt.figure(figsize=(12, 6))
plt.barh(words, freqs, color='steelblue')
plt.xlabel('Frequency')
plt.title('Top 15 Most Common Words')
plt.tight_layout()
plt.show()

## 6. Label-Specific Analysis

In [ ]:
# Word frequency by label
fake_words = []
real_words = []

for idx, row in data.iterrows():
    words = row['processed_text'].split()
    if row['label'] == 'FAKE':
        fake_words.extend(words)
    else:
        real_words.extend(words)

fake_freq = Counter(fake_words)
real_freq = Counter(real_words)

print("Most common in FAKE news:")
for word, count in fake_freq.most_common(10):
    print(f"  {word}: {count}")

print("\nMost common in REAL news:")
for word, count in real_freq.most_common(10):
    print(f"  {word}: {count}")

## 7. Data Quality Check

In [ ]:
# Check for missing values
print("Missing values:")
print(data.isnull().sum())

# Check for duplicates
duplicates = data['text'].duplicated().sum()
print(f"\nDuplicate texts: {duplicates}")

# Remove duplicates if needed
if duplicates > 0:
    data = data.drop_duplicates(subset=['text'])
    print(f"Removed duplicates. New size: {len(data)}")

## 8. Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Split data
train_data, test_data = train_test_split(
    data, test_size=0.2, random_state=42, stratify=data['label']
)

print(f"Training set size: {len(train_data)}")
print(f"Test set size: {len(test_data)}")

print(f"\nTraining set distribution:")
print(train_data['label'].value_counts())

print(f"\nTest set distribution:")
print(test_data['label'].value_counts())

## Summary

- Dataset contains balanced fake and real news articles
- Text preprocessing removes noise and normalizes input
- Vocabulary size is manageable for embedding-based models
- Ready for LSTM and BERT model training
- Data quality is good with minimal duplicates or missing values